# 부서 연결 Retriever 프로젝트 v16 - CPU/Jupyter 버전


1. Google Colab / GPU 사용하지 않음. 일반 PC Jupyter Notebook 기준.
2. 기본 도메인 문서: `업무담당규정.doc`
3. 추가 라벨링 문서: `dept.xlsx`
4. 사용자 query 입력 시 `dept.xlsx` 기준 `HNG_BR_NM` 상위 5개 출력
5. 상위 후보들에 대해 다수결 방식으로 최종 부서 1건 출력


In [2]:
!pip install rank_bm25

In [3]:
# ============================================================
# 1. 라이브러리 및 기본 설정
# ============================================================

import os
import re
import random
import subprocess
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from sklearn.feature_extraction.text import TfidfVectorizer
from rank_bm25 import BM25Okapi

from sentence_transformers import SentenceTransformer, InputExample, losses, util
from torch.utils.data import DataLoader

import torch

# GPU 사용 금지: CPU 고정
DEVICE = "cpu"

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("DEVICE:", DEVICE)


DEVICE: cpu


C:\Users\User\AppData\Local\Temp\ipykernel_9620\3336028269.py:19: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import SentenceTransformer, InputExample, losses, util


In [9]:
# ============================================================
# 2. 파일 경로 설정
# ============================================================

BASE_DOC_PATH = "./업무담당규정.docx"
DEPT_XLSX_PATH = "./dept.xlsx"

# SentenceTransformer 기본 모델
EMBEDDING_MODEL_NAME = "jhgan/ko-sroberta-multitask"


RUN_FINE_TUNE = False

FINE_TUNED_MODEL_DIR = "./fine_tuned_dept_retriever_cpu"


In [5]:
# ============================================================
# 3. dept.xlsx 로드 및 컬럼 검증
# ============================================================
# - HNG_BR_NM : 부서명
# - OFWRK_NM  : 업무 설명 / 역할 설명

def load_dept_excel(path: str) -> pd.DataFrame:
    df = pd.read_excel(path)

    required_cols = ["HNG_BR_NM", "OFWRK_NM"]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(
            f"dept.xlsx에 필요한 컬럼이 없습니다: {missing}\n"
            f"현재 컬럼: {list(df.columns)}"
        )

    df = df[required_cols].copy()
    df["HNG_BR_NM"] = df["HNG_BR_NM"].astype(str).str.strip()
    df["OFWRK_NM"] = df["OFWRK_NM"].astype(str).str.strip()
    df = df.replace({"nan": np.nan, "None": np.nan})
    df = df.dropna(subset=["HNG_BR_NM", "OFWRK_NM"]).reset_index(drop=True)

    return df

dept_df = load_dept_excel(DEPT_XLSX_PATH)

print("dept_df shape:", dept_df.shape)
display(dept_df.head())


dept_df shape: (2544, 2)


,HNG_BR_NM,OFWRK_NM
0,디지털 HR부,"[Cell장 담당업무]디지털/ICT 인사 운용, 채용, 연수 전반사항 [담당] A..."
1,디지털 HR부,"디지털/ICT 연수 기획/운영, 신입직원 연수/온보딩"
2,디지털 HR부,"디지털/ICT 연수 기획/운영, 신입직원 채용/연수/온보딩"
3,디지털 HR부,"디지털/ICT 직원 인사 [담당] Tech그룹, 정보보호본부, 기관·제휴개발부"
4,디지털 HR부,"디지털/ICT 채용, 직무 및 CDP, 신입직원 연수/온보딩"


In [10]:
# ============================================================
# 4. 업무담당규정.doc 텍스트 추출 함수
# ============================================================

def read_docx_text(docx_path: str) -> str:
    from docx import Document

    doc = Document(docx_path)
    lines = []
    for p in doc.paragraphs:
        t = p.text.strip()
        if t:
            lines.append(t)
    return "\n".join(lines)


def convert_doc_to_docx_with_word(doc_path: str) -> str:
    """Windows + MS Word 설치 환경에서 .doc -> .docx 변환"""
    import win32com.client

    doc_path = str(Path(doc_path).resolve())
    out_path = str(Path(doc_path).with_suffix(".converted.docx").resolve())

    word = win32com.client.Dispatch("Word.Application")
    word.Visible = False

    try:
        doc = word.Documents.Open(doc_path)
        # FileFormat=16은 wdFormatXMLDocument(.docx)
        doc.SaveAs(out_path, FileFormat=16)
        doc.Close()
    finally:
        word.Quit()

    return out_path


def convert_doc_to_docx_with_soffice(doc_path: str) -> str:
    """LibreOffice가 설치되어 있고 soffice 명령어가 잡힌 경우 .doc -> .docx 변환"""
    doc_path = Path(doc_path).resolve()
    out_dir = doc_path.parent

    cmd = [
        "soffice",
        "--headless",
        "--convert-to",
        "docx",
        "--outdir",
        str(out_dir),
        str(doc_path)
    ]
    subprocess.run(cmd, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)

    out_path = doc_path.with_suffix(".docx")
    if not out_path.exists():
        raise FileNotFoundError(f"LibreOffice 변환 결과 파일을 찾을 수 없습니다: {out_path}")

    return str(out_path)


def load_word_text(path: str) -> str:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"파일을 찾을 수 없습니다: {path}")

    suffix = path.suffix.lower()

    if suffix == ".docx":
        return read_docx_text(str(path))

    if suffix == ".doc":
        # 1) pywin32 + MS Word 변환 시도
        try:
            converted = convert_doc_to_docx_with_word(str(path))
            return read_docx_text(converted)
        except Exception as e_word:
            print("[경고] MS Word 변환 실패:", repr(e_word))

        # 2) LibreOffice 변환 시도
        try:
            converted = convert_doc_to_docx_with_soffice(str(path))
            return read_docx_text(converted)
        except Exception as e_soffice:
            print("[경고] LibreOffice 변환 실패:", repr(e_soffice))

        raise RuntimeError(
            ".doc 파일을 읽지 못했습니다. 해결 방법:\n"
            "1) Windows에서 MS Word + pywin32 설치 후 실행하거나\n"
            "2) LibreOffice 설치 후 soffice 명령어를 PATH에 등록하거나\n"
            "3) 업무담당규정.doc를 직접 .docx로 저장한 뒤 BASE_DOC_PATH를 .docx로 바꾸세요."
        )

    raise ValueError(f"지원하지 않는 확장자입니다: {suffix}")


base_doc_text = load_word_text(BASE_DOC_PATH)
print("업무담당규정 문서 글자 수:", len(base_doc_text))
print(base_doc_text[:500])


업무담당규정 문서 글자 수: 86787
업무담당규정
[ 일부개정2026.6.5.규정 ]
제1장 총칙
제1조(목적) 이 규정은 본부조직 및 영업조직의 담당업무를 정하는 데 그 목적이 있다. 단, 각 부서는 본 규정의 담당업무에도 불구하고, 은행의 중장기적인 가치 향상을 위한 부서간 협업에 역량과 자원을 우선 배분해야 하며, 별도의 명시 및 외규, 내규의 제약이 없는 한 본 규정의 효력 범위를 국내에 제한하지 않는다.
조문버튼
제2조(기안) 이 규정의 제정, 개폐에 기안권자는 종합기획부장으로 한다. 단, 다음과 같은 사항과 관련된 기안권자는 소관부서장으로 하되 종합기획부장의 합의를 거치도록 한다.
1. 동일 그룹 또는 본부 내 부서간 업무조정에 따른 개정
2. 타 그룹 또는 본부에 영향을 미치지 않는 담당업무의 추가, 신규에 따른 개정
3. 은행장이 별도로 결정한 사항에 따른 부수적인 개정
조문버튼
제2장 본부조직
제1절 영업추진1그룹
제3조(영업추진1부) ① 전략/기획
1. 영업추진1그룹 사업 및 재무계획 수립/실행에 관한


In [11]:
# ============================================================
# 5. 업무담당규정.doc 청크 생성 및 부서명 매핑
# ============================================================

def split_text_to_chunks(text: str, chunk_size: int = 1200, overlap: int = 200) -> list:
    """
    긴 문서를 글자 수 기준으로 chunk_size만큼 자르고 overlap만큼 겹치게 만듭니다.
    PC 환경에서 빠르게 처리하기 위한 단순 chunking입니다.
    """
    text = re.sub(r"\s+", " ", str(text)).strip()

    chunks = []
    start = 0
    chunk_id = 0

    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunk_text = text[start:end].strip()

        if chunk_text:
            chunks.append({
                "source": "업무담당규정.doc",
                "chunk_id": f"doc_{chunk_id}",
                "HNG_BR_NM": None,
                "text": chunk_text
            })

        if end >= len(text):
            break

        start += (chunk_size - overlap)
        chunk_id += 1

    return chunks


def map_doc_chunks_to_department(doc_chunks: list, dept_names: list) -> list:
    """
    문서 청크 내부에 등장하는 부서명을 찾아 HNG_BR_NM으로 매핑합니다.
    하나의 청크에 여러 부서명이 나오면 가장 먼저 등장한 부서명을 사용합니다.
    """
    mapped = []

    # 긴 부서명 우선 매칭
    dept_names_sorted = sorted(set(map(str, dept_names)), key=len, reverse=True)

    for c in doc_chunks:
        text = c["text"]
        matched = []

        for dept in dept_names_sorted:
            pos = text.find(dept)
            if pos >= 0:
                matched.append((pos, dept))

        if matched:
            matched.sort(key=lambda x: x[0])
            c = c.copy()
            c["HNG_BR_NM"] = matched[0][1]
            mapped.append(c)

    return mapped


doc_chunks_raw = split_text_to_chunks(base_doc_text, chunk_size=1200, overlap=200)
doc_chunks = map_doc_chunks_to_department(doc_chunks_raw, dept_df["HNG_BR_NM"].tolist())

print("원문 doc chunks:", len(doc_chunks_raw))
print("부서명 매핑 doc chunks:", len(doc_chunks))
pd.DataFrame(doc_chunks).head()


원문 doc chunks: 87
부서명 매핑 doc chunks: 70


,source,chunk_id,HNG_BR_NM,text
0,업무담당규정.doc,doc_0,종합기획부,업무담당규정 [ 일부개정2026.6.5.규정 ] 제1장 총칙 제1조(목적) 이 규정...
1,업무담당규정.doc,doc_1,전략영업부,소관 플랫폼사업 영업 추진 지원에 관한 사항 ④ 영업점 역량 강화 1. 그룹 내 영...
2,업무담당규정.doc,doc_2,신한Premier지원실,"문, 주선 및 대리업무에 관한 사항 ③ ESG컨설팅 1. 친환경 전환금융 목표 고객..."
3,업무담당규정.doc,doc_3,신한Premier지원실,제5조(신한Premier지원실) ① 기획 1. 은행과 증권 간 WM사업 관련 사업계...
4,업무담당규정.doc,doc_4,영업추진2부,세미나 기획 및 운영에 관한 사항 6. PWM/SFC/PIB센터 고객관리/성과관리 ...


In [12]:
# ============================================================
# 6. dept.xlsx 기반 검색 코퍼스 생성
# ============================================================


def build_dept_corpus(dept_df: pd.DataFrame) -> pd.DataFrame:
    corpus = dept_df.copy()
    corpus["source"] = "dept.xlsx"
    corpus["chunk_id"] = ["dept_" + str(i) for i in range(len(corpus))]

    # 부서명도 text에 포함하면 부서명 자체가 query에 등장할 때 잘 잡힙니다.
    corpus["text"] = corpus["HNG_BR_NM"].astype(str) + " " + corpus["OFWRK_NM"].astype(str)

    return corpus[["source", "chunk_id", "HNG_BR_NM", "text"]]


dept_corpus = build_dept_corpus(dept_df)
doc_corpus = pd.DataFrame(doc_chunks)[["source", "chunk_id", "HNG_BR_NM", "text"]] if len(doc_chunks) else pd.DataFrame(columns=["source", "chunk_id", "HNG_BR_NM", "text"])

# 최종 검색 코퍼스
# dept.xlsx를 반드시 포함하고, 업무담당규정.doc에서 부서명이 매핑된 청크만 추가합니다.
corpus_df = pd.concat([dept_corpus, doc_corpus], ignore_index=True)
corpus_df = corpus_df.dropna(subset=["HNG_BR_NM", "text"]).reset_index(drop=True)

print("dept corpus:", len(dept_corpus))
print("mapped doc corpus:", len(doc_corpus))
print("total corpus:", len(corpus_df))
display(corpus_df.head())


dept corpus: 2544
mapped doc corpus: 70
total corpus: 2614


,source,chunk_id,HNG_BR_NM,text
0,dept.xlsx,dept_0,디지털 HR부,"디지털 HR부 [Cell장 담당업무]디지털/ICT 인사 운용, 채용, 연수 전반사항..."
1,dept.xlsx,dept_1,디지털 HR부,"디지털 HR부 디지털/ICT 연수 기획/운영, 신입직원 연수/온보딩"
2,dept.xlsx,dept_2,디지털 HR부,"디지털 HR부 디지털/ICT 연수 기획/운영, 신입직원 채용/연수/온보딩"
3,dept.xlsx,dept_3,디지털 HR부,"디지털 HR부 디지털/ICT 직원 인사 [담당] Tech그룹, 정보보호본부, 기관..."
4,dept.xlsx,dept_4,디지털 HR부,"디지털 HR부 디지털/ICT 채용, 직무 및 CDP, 신입직원 연수/온보딩"


In [13]:
# ============================================================
# 7. dept.xlsx 라벨 데이터로 SentenceTransformer CPU 파인튜닝
# ============================================================

def build_department_profiles(dept_df: pd.DataFrame) -> dict:
    profiles = (
        dept_df
        .groupby("HNG_BR_NM")["OFWRK_NM"]
        .apply(lambda s: " ".join(s.astype(str).tolist()))
        .to_dict()
    )
    return profiles


def fine_tune_retriever_cpu(
    dept_df: pd.DataFrame,
    base_model_name: str,
    output_dir: str,
    epochs: int = 1,
    batch_size: int = 16,
    lr: float = 2e-5
):
    profiles = build_department_profiles(dept_df)

    train_examples = []
    for _, row in dept_df.iterrows():
        dept = str(row["HNG_BR_NM"])
        query_like_text = str(row["OFWRK_NM"])
        positive_context = f"{dept} {profiles[dept]}"

        train_examples.append(
            InputExample(texts=[query_like_text, positive_context])
        )

    model = SentenceTransformer(base_model_name, device=DEVICE)
    train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=batch_size)
    train_loss = losses.MultipleNegativesRankingLoss(model)

    model.fit(
        train_objectives=[(train_dataloader, train_loss)],
        epochs=epochs,
        warmup_steps=max(10, int(len(train_dataloader) * 0.1)),
        optimizer_params={"lr": lr},
        show_progress_bar=True
    )

    model.save(output_dir)
    return model


if RUN_FINE_TUNE:
    embedder = fine_tune_retriever_cpu(
        dept_df=dept_df,
        base_model_name=EMBEDDING_MODEL_NAME,
        output_dir=FINE_TUNED_MODEL_DIR,
        epochs=1,
        batch_size=16,
        lr=2e-5
    )
else:
    embedder = SentenceTransformer(EMBEDDING_MODEL_NAME, device=DEVICE)

print("임베더 준비 완료:", embedder)


임베더 준비 완료: SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'RobertaModel'})
  (1): Pooling({'embedding_dimension': 768, 'pooling_mode': 'mean', 'include_prompt': True})
)


In [14]:
# ============================================================
# 8. 임베딩 / TF-IDF / BM25 인덱스 생성
# ============================================================

def simple_tokenize(text: str) -> list:
    """한국어 형태소 분석기 없이 PC에서 바로 돌아가는 단순 토크나이저"""
    return re.findall(r"[가-힣A-Za-z0-9]+", str(text).lower())


def normalize_matrix(x: np.ndarray) -> np.ndarray:
    return x / (np.linalg.norm(x, axis=1, keepdims=True) + 1e-12)


# 1) Dense embedding
corpus_texts = corpus_df["text"].astype(str).tolist()

corpus_embs = embedder.encode(
    corpus_texts,
    convert_to_numpy=True,
    show_progress_bar=True,
    batch_size=64
)
corpus_embs = normalize_matrix(corpus_embs)

# 2) TF-IDF
tfidf_vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(2, 5),
    min_df=1
)
tfidf_matrix = tfidf_vectorizer.fit_transform(corpus_texts)

# 3) BM25
tokenized_corpus = [simple_tokenize(t) for t in corpus_texts]
bm25 = BM25Okapi(tokenized_corpus)

print("Dense embeddings:", corpus_embs.shape)
print("TF-IDF matrix:", tfidf_matrix.shape)
print("BM25 corpus:", len(tokenized_corpus))


Batches:   0%|          | 0/41 [00:00<?, ?it/s]

Dense embeddings: (2614, 768)
TF-IDF matrix: (2614, 118746)
BM25 corpus: 2614


In [15]:
# ============================================================
# 9. Query → HNG_BR_NM 상위 5개 + 다수결 최종 1건
# ============================================================

def minmax_norm(x):
    x = np.asarray(x, dtype=float)
    return (x - x.min()) / (np.ptp(x) + 1e-12)


def retrieve_scored_rows(
    query: str,
    candidate_k: int = 50,
    dense_weight: float = 0.50,
    tfidf_weight: float = 0.30,
    bm25_weight: float = 0.20
) -> pd.DataFrame:
    """
    query에 대해 dense / TF-IDF / BM25 점수를 계산하고 상위 candidate_k 행을 반환합니다.
    """
    candidate_k = min(candidate_k, len(corpus_df))

    # 1) Dense cosine similarity
    q_emb = embedder.encode([query], convert_to_numpy=True)
    q_emb = q_emb / (np.linalg.norm(q_emb, axis=1, keepdims=True) + 1e-12)
    dense_scores_all = (corpus_embs @ q_emb[0])

    # dense 기준 후보를 먼저 넉넉히 뽑음
    dense_top_idx = np.argsort(dense_scores_all)[-candidate_k:][::-1]

    # 2) TF-IDF 점수
    q_tfidf = tfidf_vectorizer.transform([query])
    tfidf_scores_all = (tfidf_matrix @ q_tfidf.T).toarray().ravel()

    # 3) BM25 점수
    bm25_scores_all = np.asarray(bm25.get_scores(simple_tokenize(query)))

    # 후보 점수만 추출
    d = dense_scores_all[dense_top_idx]
    t = tfidf_scores_all[dense_top_idx]
    b = bm25_scores_all[dense_top_idx]

    final_scores = (
        dense_weight * minmax_norm(d) +
        tfidf_weight * minmax_norm(t) +
        bm25_weight * minmax_norm(b)
    )

    result = corpus_df.iloc[dense_top_idx].copy()
    result["dense_score"] = d
    result["tfidf_score"] = t
    result["bm25_score"] = b
    result["score"] = final_scores

    result = result.sort_values("score", ascending=False).reset_index(drop=True)
    return result


def predict_department(
    query: str,
    top_n_departments: int = 5,
    candidate_k: int = 50,
    vote_pool_size: int = 15
):
    """
    1) query와 유사한 후보 row를 candidate_k개 검색
    2) 최종 점수 순으로 vote_pool_size개를 투표 풀로 사용
    3) HNG_BR_NM 다수결로 상위 5개 부서와 최종 1개 부서를 결정
    """
    scored_rows = retrieve_scored_rows(query, candidate_k=candidate_k)

    vote_pool_size = min(vote_pool_size, len(scored_rows))
    vote_pool = scored_rows.head(vote_pool_size).copy()

    # 부서별 다수결 + 점수 집계
    summary = (
        vote_pool
        .groupby("HNG_BR_NM")
        .agg(
            vote_count=("HNG_BR_NM", "size"),
            score_sum=("score", "sum"),
            score_max=("score", "max"),
            best_text=("text", "first"),
            best_source=("source", "first")
        )
        .reset_index()
    )

    # 다수결 우선, 동률이면 score_sum / score_max 순
    summary = summary.sort_values(
        ["vote_count", "score_sum", "score_max"],
        ascending=[False, False, False]
    ).reset_index(drop=True)

    top5 = summary.head(top_n_departments).copy()
    final_department = top5.iloc[0]["HNG_BR_NM"] if len(top5) else None

    return {
        "query": query,
        "final_department": final_department,
        "top5_departments": top5,
        "scored_rows": scored_rows,
        "vote_pool": vote_pool
    }


def print_prediction(result: dict):
    print("[고객 문의]")
    print(result["query"])
    print("\n[HNG_BR_NM 상위 5개 - 다수결 기준]")

    top5 = result["top5_departments"]
    if len(top5) == 0:
        print("검색 결과 없음")
        return

    for i, row in top5.iterrows():
        print(
            f"{i+1}. {row['HNG_BR_NM']} "
            f"| votes={int(row['vote_count'])} "
            f"| score_sum={row['score_sum']:.4f} "
            f"| score_max={row['score_max']:.4f}"
        )

    print("\n[최종 출력 1건]")
    print(result["final_department"])


In [34]:
# ============================================================
# 10. 단건 테스트
# ============================================================

with open('query1.txt', 'r', encoding='utf-8') as file:
    query1 = file.read() 


print(query1)

“마이데이터 활용 금리인하요구 서비스” 등 혁신금융서비스 3건 신규 지정 의결 -금번 신규 지정으로 혁신금융서비스 누적 지정건수는 총 1,075건
2026-06-17 조회수 : 532
담당부서디지털금융총괄과 담당자송현지 서기관 연락처02-2100-2841
담당부서디지털금융총괄과 담당자이혜인 사무관 연락처02-2100-2872


“마이데이터 활용 금리인하요구 서비스” 등

 

혁신금융서비스 3건 신규 지정 의결



-금번 신규 지정으로 혁신금융서비스 누적 지정건수는 총 1,075건





금융위원회(위원장 이억원)는 6월 17일 정례회의를 통해 3건의 혁신금융서비스를 신규로 지정하였다. 이로써 현재까지 총 1,075건의 서비스가 혁신금융서비스로 지정되어 시장에서 테스트할 수 있게 되었다.



혁신금융서비스 지정 주요 내용은 다음과 같다.(“금융위 의결 결과 세부내용” : 참고)



금융위원회는 농업협동조합중앙회의 ‘마이데이터 활용 금리인하요구 서비스’를 혁신금융서비스로 신규 지정하였다. 이에 따라 농협협동조합중앙회가 제공하는 마이데이터서비스를 활용하여 적시에 금리인하요구권을 행사할 수 있게 되는 바 대출이자 부담 절감, 이용자의 시간·노력 감소 등 금융소비자 편익이 증가할 것으로 기대된다.



 또한, SK플래닛 및 전북은행의 ‘OK 캐시백 제휴통장 서비스’도 혁신금융서비스로 신규 지정하였다. SK플래닛이 전북은행 제휴 통장 개설을 중개하고 제휴계좌 상품을 광고할 수 있도록 규제 특례를 부여하였다. 이를 통해 OK캐쉬백 앱 이용자는 예금자보호가 적용되는 전북은행 제휴계좌에 SK플래닛이 발행한 선불전자지급수단을 보관 할 수 있게 되고 예금이자 및 제휴혜택 등을 제공받는 등 소비자 효용이 제고된다.



더불어, KB자산운용의 ‘개인형 퇴직연금 로보어드바이저 일임서비스’가 혁신금융서비스로 신규 지정되었다. 로보어드바이저가 알고리즘 등을 통해 투자자 성향별 맞춤형 포트폴리오를 자동 생성하고 개인형퇴직연금(IRP) 적립금에 대한 일임운용서비스를 제공하게 된

In [35]:

result = predict_department(
    query=query1,
    top_n_departments=5,
    candidate_k=50,
    vote_pool_size=15
)

print_prediction(result)

# 상세 후보 row 확인
display(result["scored_rows"].head(10)[["HNG_BR_NM", "source", "score", "dense_score", "tfidf_score", "bm25_score", "text"]])


[고객 문의]
“마이데이터 활용 금리인하요구 서비스” 등 혁신금융서비스 3건 신규 지정 의결 -금번 신규 지정으로 혁신금융서비스 누적 지정건수는 총 1,075건
2026-06-17 조회수 : 532
담당부서디지털금융총괄과 담당자송현지 서기관 연락처02-2100-2841
담당부서디지털금융총괄과 담당자이혜인 사무관 연락처02-2100-2872


“마이데이터 활용 금리인하요구 서비스” 등

 

혁신금융서비스 3건 신규 지정 의결



-금번 신규 지정으로 혁신금융서비스 누적 지정건수는 총 1,075건





금융위원회(위원장 이억원)는 6월 17일 정례회의를 통해 3건의 혁신금융서비스를 신규로 지정하였다. 이로써 현재까지 총 1,075건의 서비스가 혁신금융서비스로 지정되어 시장에서 테스트할 수 있게 되었다.



혁신금융서비스 지정 주요 내용은 다음과 같다.(“금융위 의결 결과 세부내용” : 참고)



금융위원회는 농업협동조합중앙회의 ‘마이데이터 활용 금리인하요구 서비스’를 혁신금융서비스로 신규 지정하였다. 이에 따라 농협협동조합중앙회가 제공하는 마이데이터서비스를 활용하여 적시에 금리인하요구권을 행사할 수 있게 되는 바 대출이자 부담 절감, 이용자의 시간·노력 감소 등 금융소비자 편익이 증가할 것으로 기대된다.



 또한, SK플래닛 및 전북은행의 ‘OK 캐시백 제휴통장 서비스’도 혁신금융서비스로 신규 지정하였다. SK플래닛이 전북은행 제휴 통장 개설을 중개하고 제휴계좌 상품을 광고할 수 있도록 규제 특례를 부여하였다. 이를 통해 OK캐쉬백 앱 이용자는 예금자보호가 적용되는 전북은행 제휴계좌에 SK플래닛이 발행한 선불전자지급수단을 보관 할 수 있게 되고 예금이자 및 제휴혜택 등을 제공받는 등 소비자 효용이 제고된다.



더불어, KB자산운용의 ‘개인형 퇴직연금 로보어드바이저 일임서비스’가 혁신금융서비스로 신규 지정되었다. 로보어드바이저가 알고리즘 등을 통해 투자자 성향별 맞춤형 포트폴리오를 자동 생성하고 개인형퇴직연금(IRP) 적립금에 대한 일임운용서비스

,HNG_BR_NM,source,score,dense_score,tfidf_score,bm25_score,text
0,여신기획부,업무담당규정.doc,0.659865,0.633950,0.104829,27.072494,"사항 3. 고객의 소리, 고객경험Data기반 업무/제도 개선방안 도출 및 개선 추진..."
1,모형공학부,dept.xlsx,0.621375,0.662697,0.031581,8.845799,모형공학부 - 소매 모형 바젤 II 내부등급법 승인 업무 - 가계대출모형 운영 및 ...
2,디지털금융센터(총괄),dept.xlsx,0.614022,0.646782,0.034750,34.712290,"디지털금융센터(총괄) ""[비대면신규팀]_운영스탭 ★리테일중심영업점/ 기금OPC/비대..."
3,부동산금융부,업무담당규정.doc,0.535840,0.635962,0.069603,19.068056,"관한 사항 3. 상기 업무 관련 조건변경, Covenant/승인조건 관리, 추가인출..."
4,고객경험혁신센터,업무담당규정.doc,0.530475,0.616312,0.108585,31.623920,터 사업 기획 총괄 및 정책 대응에 관한 사항 2. 전행 공공마이데이터 활용 정책 ...
5,프로젝트금융부,업무담당규정.doc,0.526669,0.633376,0.070893,21.249168,"ine, 대출 등)에 관한 사항 3. 상기 업무 관련 조건변경, Covenant/승..."
6,기업여신지원부,업무담당규정.doc,0.488209,0.636210,0.061435,13.879528,"개선업무의 기획, 운영에 관한 사항 1. 기업개선 제도 기획 및 지원에 관한 사항 ..."
7,IB/글로벌심사부,업무담당규정.doc,0.397371,0.621401,0.070631,17.754625,여신에 대한 심사 및 의사 결정에 관한 사항 나. 차주/기업여신 모니터링 및 모니터...
8,기업여신심사부,업무담당규정.doc,0.372176,0.619519,0.066540,18.564182,개선 및 여신역량 강화 지원에 관한 사항 5. 글로벌 자산건전성(연체 포함) 지표 ...
9,모형공학부,dept.xlsx,0.354170,0.638776,0.025255,3.428839,모형공학부 - 소매 모형 바젤 II 내부등급법 승인 업무 - 가계대출모형 운영 및 ...


In [ ]:
# # ============================================================
# # 11. 여러 query 배치 예측
# # ============================================================
# # queries 리스트에 고객 문의를 여러 개 넣으면 결과 DataFrame으로 반환합니다.

# def batch_predict(queries: list, top_n_departments: int = 5, candidate_k: int = 50, vote_pool_size: int = 15) -> pd.DataFrame:
#     rows = []

#     for q in tqdm(queries, desc="predict"):
#         r = predict_department(
#             query=q,
#             top_n_departments=top_n_departments,
#             candidate_k=candidate_k,
#             vote_pool_size=vote_pool_size
#         )

#         top5 = r["top5_departments"]["HNG_BR_NM"].tolist()
#         row = {
#             "query": q,
#             "final_department": r["final_department"]
#         }

#         for i in range(top_n_departments):
#             row[f"top{i+1}"] = top5[i] if i < len(top5) else None

#         rows.append(row)

#     return pd.DataFrame(rows)


# sample_queries = [
#     """""",
#     """"""",",
#     """"""",",
#     """"""",",
#     """"""","
# ]

# batch_result = batch_predict(sample_queries)
# display(batch_result)


In [ ]:
# # ============================================================
# # 12. 평가 데이터가 있을 경우 정확도 계산
# # ============================================================
# # - 문장
# # - 유관부서
# #
# # TEST_XLSX_PATH 값을 실제 파일 경로로 바꿔 실행하

# TEST_XLSX_PATH = "./test_set.xlsx"

# def evaluate_if_exists(test_path: str):
#     if not Path(test_path).exists():
#         print(f"평가 파일 없음: {test_path}")
#         return None

#     test_df = pd.read_excel(test_path)
#     required = ["문장", "유관부서"]
#     missing = [c for c in required if c not in test_df.columns]
#     if missing:
#         raise ValueError(f"평가 파일에 필요한 컬럼이 없습니다: {missing}")

#     pred_df = batch_predict(test_df["문장"].astype(str).tolist())
#     out = pd.concat([test_df.reset_index(drop=True), pred_df.drop(columns=["query"])], axis=1)

#     out["correct_top1"] = out["final_department"].astype(str) == out["유관부서"].astype(str)
#     top_cols = ["top1", "top2", "top3", "top4", "top5"]
#     out["correct_top5"] = out.apply(
#         lambda r: str(r["유관부서"]) in [str(r[c]) for c in top_cols],
#         axis=1
#     )

#     print("Top-1 accuracy:", out["correct_top1"].mean())
#     print("Top-5 accuracy:", out["correct_top5"].mean())

#     return out

# # eval_result = evaluate_if_exists(TEST_XLSX_PATH)
# # display(eval_result.head())
